# Analisis Kesesuaian Star Rating dan Sentimen Ulasan
## Perbandingan Algoritma: Multinomial NB, Linear SVM, & Logistic Regression (Elite Edition)

Proyek skripsi ini melakukan analisis mendalam terhadap sentimen ulasan Google Play Store. 
**Fokus Penelitian:**
1. Membangun model klasifikasi sentimen yang tangguh.
2. Membandingkan performa tiga algoritma (Naive Bayes, SVM, Logistic Regression).
3. Menganalisis *mismatch* antara rating bintang pengguna dan sentimen teks ulasan.
4. Menyediakan data yang kaya untuk visualisasi di Tableau.

### 1. Persiapan Lingkungan dan Import Library
Tahap ini mencakup pemuatan pustaka untuk Machine Learning, Visualisasi, dan Explainability (LIME).

In [ ]:
import pandas as pd
import numpy as np
import re
import kagglehub
from kagglehub import KaggleDatasetAdapter
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# ML Core
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, 
    balanced_accuracy_score, f1_score, roc_auc_score, 
    precision_recall_curve, auc, roc_curve
)
from imblearn.over_sampling import RandomOverSampler

# Visualisasi Lanjut
import scikitplot as skplt
from lime import lime_text
from lime.make_pipeline import make_pipeline

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

# Resources
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Set style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

### 2. Pemuatan Dataset
Data diambil dari Google Play App Reviews di Kaggle.

In [ ]:
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "hassaanmustafavi/google-play-app-reviews-dataset",
  "GooglePlay_App_Data.csv",
)

print(f"Total data awal: {len(df)}")
df.head()

### 3. Pembersihan Data & Pelabelan Sentimen
Kita memetakan rating 1-2 sebagai negatif, 3 netral, dan 4-5 positif.

In [ ]:
df = df.dropna(subset=['review_description'])
df = df.rename(columns={
    "review_description": "review",
    "rating": "star_rating"
})

# Pelabelan
df['sentiment'] = df['star_rating'].apply(lambda x:
    'positive' if x >= 4 else
    'neutral' if x == 3 else
    'negative'
)

# Statistik dasar ulasan
df['review_length'] = df['review'].apply(len)
df['word_count'] = df['review'].apply(lambda x: len(x.split()))

print("Statistik Distribusi Sentimen:")
print(df['sentiment'].value_counts())

plt.figure(figsize=(8, 5))
sns.countplot(x='sentiment', data=df, order=['negative', 'neutral', 'positive'])
plt.title('Distribusi Sentimen Berdasarkan Rating Bintang')
plt.show()

### 4. Text Preprocessing (Lemmatization)
Proses pembersihan teks ulasan.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    words = text.split()
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return " ".join(words)

print("Preprocessing sedang berjalan...")
df['clean_review'] = df['review'].apply(clean_text)
print("Preprocessing selesai.")

### 5. Feature Extraction & Data Split
Menggunakan TF-IDF dengan N-grams (1,2).

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X = vectorizer.fit_transform(df['clean_review'])
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Handling Class Imbalance
ros = RandomOverSampler(random_state=42)
X_train_res, y_train_res = ros.fit_resample(X_train, y_train)

### 6. Perbandingan Algoritma (The Battle)
Kita akan melatih tiga model berbeda dan membandingkan hasilnya.

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, multi_class='multinomial'),
    "Linear SVM": LinearSVC(dual=False)
}

results = []

for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train_res, y_train_res)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    bal_acc = balanced_accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    
    results.append({
        "Model": name,
        "Accuracy": acc,
        "Balanced Accuracy": bal_acc,
        "F1-Score (Macro)": f1
    })

df_results = pd.DataFrame(results)
df_results

### 7. Visualisasi Komparatif
Menyajikan perbandingan performa secara visual.

In [ ]:
df_results_melted = df_results.melt(id_vars="Model", var_name="Metric", value_name="Score")
sns.barplot(x='Metric', y='Score', hue='Model', data=df_results_melted)
plt.title('Perbandingan Performa Model Sentimen')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

### 8. Analisis Kesalahan (Error Analysis)
Melihat di mana model (misal: SVM) paling sering salah menebak.

In [ ]:
best_model = models["Logistic Regression"] # Logistic Regression biasanya stabil
y_pred_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, y_pred_best)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['negative', 'neutral', 'positive'], 
            yticklabels=['negative', 'neutral', 'positive'])
plt.title('Confusion Matrix: Logistic Regression')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

### 9. Interpretasi Model (Feature Importance)
Melihat kata-kata yang paling dominan untuk setiap sentimen.

In [ ]:
def plot_coefficients(model, vectorizer, top_features=20):
    coef = model.coef_[0] if len(model.coef_) == 1 else model.coef_[2] # Koefisien untuk kelas positif
    top_positive_coefficients = np.argsort(coef)[-top_features:]
    top_negative_coefficients = np.argsort(coef)[:top_features]
    top_coefficients = np.hstack([top_negative_coefficients, top_positive_coefficients])
    
    plt.figure(figsize=(15, 5))
    colors = ['red' if c < 0 else 'blue' for c in coef[top_coefficients]]
    plt.bar(np.arange(2 * top_features), coef[top_coefficients], color=colors)
    feature_names = np.array(vectorizer.get_feature_names_out())
    plt.xticks(np.arange(2 * top_features), feature_names[top_coefficients], rotation=60, ha='right')
    plt.title("Top Words for Negative (Red) vs Positive (Blue) Sentiment")
    plt.show()

plot_coefficients(models["Logistic Regression"], vectorizer)

### 10. Persiapan Data untuk Tableau
Kita mengekspor data yang kaya fitur untuk diolah di Tableau.

In [ ]:
# Ambil subset data test untuk analisis detail
test_indices = df.iloc[y_test.index].index
export_df = df.loc[test_indices].copy()

export_df['predicted_sentiment'] = y_pred_best
export_df['is_mismatch'] = export_df['sentiment'] != export_df['predicted_sentiment']

# Probabilitas (Confidence Score) jika tersedia
if hasattr(best_model, "predict_proba"):
    probs = best_model.predict_proba(X_test)
    export_df['confidence_score'] = probs.max(axis=1)
else:
    export_df['confidence_score'] = 1.0

export_df.to_csv("tableau_sentiment_analysis.csv", index=False)
print("File 'tableau_sentiment_analysis.csv' berhasil diekspor untuk Tableau!")

### 11. Kesimpulan Sementara
Berdasarkan analisis di atas, model perbandingan memberikan gambaran komprehensif mengenai perilaku sentimen pengguna. Model Logistic Regression/SVM cenderung lebih stabil dibanding Naive Bayes murni pada dataset ini. Fenomena *mismatch* dapat dianalisis lebih lanjut di Tableau untuk melihat korelasi antara panjang ulasan dan ketidaksesuaian rating.